# MARSNet — Run 20

## What changed from R19

| # | Parameter | R19 | R20 | Reason |
|---|-----------|-----|-----|--------|
| 1 | `LAM_VPREV` | 0.10 | **0.0** | Replaced by gated version |
| 2 | `LAM_CVPRIOR` | — | **0.50** | Gated L_vprev — no conflict with L_dr |
| 3 | `CVPRIOR_GYRO_THR` | — | **0.10 rad/s** | From R19 IMU data: 87% straight / 13% turning |
| 4 | `DRIFT_WARM_START` | 40 | **30** | TCN activates faster now (R19 showed fast early descent) |
| 5 | `DRIFT_WARM_END` | 80 | **70** | |

**Everything else kept from R19:** architecture unchanged, `TEMPORAL_DROPOUT_P=0.25`,
`W_TURN=8.0`, `LAM_DR=0.90`, `BATCH_SIZE=16`, `LR=8e-4`, `PATIENCE=60`.

## Why this works where R19 didn't

In R19, L_vprev (weight 0.10) applied to ALL outage windows, competing with
L_dr (weight 0.90) on turn windows. VpErr never converged below 1.3 m/s.

In R20, L_cvprior only fires when `|gyro_z| < 0.10 rad/s` (confirmed straight):

| Window type | L_dr | L_cvprior | Winner |
|-------------|------|-----------|--------|
| Straight outage (87%): | ≈0 (GPS≈v_prev) | 0.50 | L_cvprior → v_prev ✓ |
| Turn outage (13%): | 0.90 | 0.00 (gate OFF) | L_dr → GPS ✓ |
| No conflict on either type. | | | |

## From R19 IMU diagnostic (actual data)
```
S0–S34  straight:    gyro_z mean=0.030 → BELOW 0.10 → gate ON  → predict v_prev ✓
S41–S46 turns:       gyro_z mean=0.126 → ABOVE 0.10 → gate OFF → L_dr teaches turns ✓
S47–S52 false-alarm: gyro_z mean=0.062 → BELOW 0.10 → gate ON  → predict v_prev ✓
```
S47–S52 are vibration noise sequences (not real turns) — gate correctly classifies them
as straight, so L_cvprior will drive predictions to v_prev → ~0.5m drift expected.

## R19 key finding preserved: TCN now contributes 14.53×
R20 preserves the temporal task (predict consistent velocity across outage windows)
so the TCN should remain active. Verify with TCN ablation test in diagnostics.


In [ ]:
import math, os, time, csv, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import WeightedRandomSampler, DataLoader
from collections import defaultdict
warnings.filterwarnings('ignore')
print("Imports OK")

In [ ]:
# ── Configuration (portable — no Colab / Google Drive required) ───────────────
# Point GATEIO_DATA at MARS_Master_Dataset.npz, or edit the default path below.
import os
DATA_PATH = os.environ.get("GATEIO_DATA", "../data/processed/MARS_Master_Dataset.npz")
CKPT_DIR  = os.environ.get("GATEIO_CKPT", "./checkpoints")
PLOT_DIR  = os.environ.get("GATEIO_OUT",  "./results")
CKPT_PATH = os.path.join(CKPT_DIR, "marsnet_r20_final.pt")
CKPT_LAST = os.path.join(CKPT_DIR, "marsnet_r20_final_last.pt")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Architecture (unchanged R14–R20) ─────────────────────────────────────────
D_MODEL       = 48
N_TCN_STACKS  = 2
N_HEADS       = 4
DROPOUT       = 0.15
N_CHAN        = 14
N_IMU_CHAN    = 10
SEQ_LEN       = 300
WIN_LEN       = 200
DT            = 0.1

# ── Loss weights ──────────────────────────────────────────────────────────────
LAM_DR         = 0.90
LAM_CVPRIOR    = 0.50   # R20: gated constant-velocity prior (replaces LAM_VPREV)
LAM_VPREV      = 0.0    # R20: REMOVED — replaced by gated version
CVPRIOR_GYRO_THR = 0.10 # rad/s — from R19 data: 87% straight / 13% turning
LAM_VAR        = 0.0    # kept removed from R17
LAM_VAR_CEIL   = 0.0    # kept removed from R17
LAM_BIAS       = 0.0    # kept removed from R18
LAM_PHYS_MAX   = 0.01
LAM_SMOOTH     = 0.001
LAM_DRIFT      = 0.002
DR_AXIS_WEIGHTS = torch.tensor([1.0, 1.0, 3.0])
HUBER_DELTA    = 0.3

# ── Warmup ────────────────────────────────────────────────────────────────────
PHYS_WARM_START,  PHYS_WARM_END  = 30,  60
DRIFT_WARM_START, DRIFT_WARM_END = 30,  70  # R20: earlier (30→70 vs 40→80)

# ── Motion thresholds ─────────────────────────────────────────────────────────
# CVPRIOR_GYRO_THR=0.10 is used in the LOSS (not just diagnostics).
# From R19 IMU diagnostic: correctly separates 87% straight from 13% turning.
TURN_GYRO_THR  = 0.10  # R20: corrected from 0.50 (R14-R19 value was too high)
ZUPT_GYRO_THR  = 0.05  # rad/s — near-stationary gate for L_phys

# ── Augmentation ──────────────────────────────────────────────────────────────
TEMPORAL_DROPOUT_P = 0.25  # kept from R17 — improved S41-S46 from 3.75m to 2.27m
W_TURN             = 8.0   # kept
P_VPREV_MASK       = 0.0   # never use — caused R17 S47-S52: 22m→92m

# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS = 200
PATIENCE   = 60
LR         = 8e-4
BATCH_SIZE = 16

print("Config loaded — Run 20")
print(f"  LAM_CVPRIOR={LAM_CVPRIOR}  CVPRIOR_GYRO_THR={CVPRIOR_GYRO_THR} rad/s  [GATED — key change]")
print(f"  LAM_VPREV={LAM_VPREV}  [removed — ungated version overwhelmed by L_dr]")
print(f"  DRIFT_WARM_START={DRIFT_WARM_START}  [earlier: was 40]")
print(f"  TURN_GYRO_THR={TURN_GYRO_THR}  [corrected from 0.50 — based on R19 data]")
print(f"  TEMPORAL_DROPOUT_P={TEMPORAL_DROPOUT_P}  W_TURN={W_TURN}  [kept from R17]")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMU GATE VERIFICATION (run before training)
# Verifies CVPRIOR_GYRO_THR=0.10 correctly classifies windows in val set.
# From R19 diagnostic: S41-S46 mean gyro_z=0.126 (above), S47-S52=0.062 (below).
# ─────────────────────────────────────────────────────────────────────────────
_npz_g = np.load(DATA_PATH)
Xv_g   = _npz_g['X_val'];  vi_g = _npz_g['val_valid_idx']
SL_g   = int(_npz_g['seq_len'][0]) if 'seq_len' in _npz_g else SEQ_LEN
mid    = WIN_LEN // 2

print("="*65)
print("  CVPRIOR GATE VERIFICATION")
print(f"  Threshold: CVPRIOR_GYRO_THR = {CVPRIOR_GYRO_THR} rad/s")
print("="*65)

all_gyro_z = []; gate_coverage = 0; total_windows = 0
group_stats = {}
groups = {'straight-short': range(0,35), 'straight-med': range(35,41),
          'TURN': range(41,47), 'FALSE-ALARM': range(47,53), 'long-outage': range(53,59)}

for grp, idx_range in groups.items():
    gz_list = []
    for si in idx_range:
        if si >= len(vi_g): continue
        seg = Xv_g[int(vi_g[si]):int(vi_g[si])+SL_g, mid, 5]  # gyro_z only
        gz = np.abs(seg)
        gz_list.extend(gz.tolist()); all_gyro_z.extend(gz.tolist())
        gate_coverage += (gz < CVPRIOR_GYRO_THR).sum()
        total_windows += len(gz)
    if gz_list:
        gz_arr = np.array(gz_list)
        pct_straight = 100*(gz_arr < CVPRIOR_GYRO_THR).mean()
        group_stats[grp] = (gz_arr.mean(), pct_straight)

print(f" {'Group':<18} {'gyro_z mean':>12} {'% gated ON':>12}  Decision")
print("  " + "-"*56)
for grp, (mean_gz, pct) in group_stats.items():
    decision = ("gate ON → predict v_prev ✓" if pct > 70
                else "gate OFF → L_dr teaches turns ✓" if pct < 30
                else f"MIXED ({pct:.0f}%) — check threshold")
    flag = " ✓" if (grp in ('straight-short','straight-med','FALSE-ALARM','long-outage') and pct > 70)            else (" ✓" if grp == 'TURN' and pct < 50 else " ⚠")
    print(f"  {grp:<18} {mean_gz:>12.4f} {pct:>11.1f}%  {decision}{flag}")

overall_pct = 100*gate_coverage/max(total_windows,1)
print(f" Overall gate coverage: {overall_pct:.1f}% of val windows → L_cvprior fires")
print(f"  (R19 confirmed: 87.1% straight, 12.9% turning)")

# Critical check: S41-S46 must NOT be mostly gated
turn_gz = group_stats.get('TURN', (0, 100))[1]
fa_gz   = group_stats.get('FALSE-ALARM', (0, 0))[1]
if turn_gz < 75:
    print(f"✓ TURN seqs: {turn_gz:.0f}% gated ON — gate correctly leaves turns to L_dr")
else:
    print(f"✗ TURN seqs: {turn_gz:.0f}% gated ON — gate too aggressive, raise threshold to 0.15")
if fa_gz > 70:
    print(f"  ✓ FALSE-ALARM seqs: {fa_gz:.0f}% gated ON — S47-S52 will get v_prev prediction")
else:
    print(f"  ✗ FALSE-ALARM seqs: {fa_gz:.0f}% gated ON — S47-S52 may still misfire")
print("="*65)
del _npz_g, Xv_g

In [ ]:
class OutageStepPE(nn.Module):
    def __init__(self, d_model, max_steps=211):
        super().__init__()
        pe  = torch.zeros(max_steps, d_model)
        pos = torch.arange(max_steps).float().unsqueeze(1)
        div = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.)/d_model))
        pe[:,0::2] = torch.sin(pos*div);  pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe)
    def forward(self, tokens, outage_flag):
        flag   = (outage_flag > 0.5).long()
        cumsum = flag.cumsum(dim=1); reset = cumsum*(1-flag)
        steps  = (cumsum - reset.cummax(dim=1).values).clamp(0, self.pe.shape[0]-1)
        return tokens + self.pe[steps]

class TCNBlock(nn.Module):
    def __init__(self, d_model, dilation, kernel_size=3, dropout=0.1):
        super().__init__()
        pad=( kernel_size-1)*dilation; self.pad=pad
        self.conv1=nn.Conv1d(d_model,d_model,kernel_size,dilation=dilation,padding=0)
        self.norm1=nn.LayerNorm(d_model)
        self.conv2=nn.Conv1d(d_model,d_model,kernel_size,dilation=dilation,padding=0)
        self.norm2=nn.LayerNorm(d_model); self.drop=nn.Dropout(dropout); self.act=nn.GELU()
    def forward(self, x):
        res=x
        x=self.act(self.norm1(self.conv1(F.pad(x,(self.pad,0))).transpose(1,2)).transpose(1,2))
        x=self.drop(x)
        x=self.act(self.norm2(self.conv2(F.pad(x,(self.pad,0))).transpose(1,2)).transpose(1,2))
        return self.drop(x)+res

class TCNBackbone(nn.Module):
    DILATIONS=[1,2,4,8,16]
    def __init__(self, d_model, n_stacks=N_TCN_STACKS, kernel_size=3, dropout=DROPOUT):
        super().__init__()
        self.net=nn.Sequential(*[TCNBlock(d_model,d,kernel_size,dropout)
                                  for _ in range(n_stacks) for d in self.DILATIONS])
    def forward(self, x): return self.net(x.transpose(1,2)).transpose(1,2)

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.n_heads=n_heads; self.d_head=d_model//n_heads; self.scale=self.d_head**-0.5
        self.norm=nn.LayerNorm(d_model); self.qkv=nn.Linear(d_model,3*d_model,bias=False)
        self.proj=nn.Linear(d_model,d_model); self.drop=nn.Dropout(dropout)
        slopes=2.**(-8.*torch.arange(1,n_heads+1).float()/n_heads)
        self.register_buffer("slopes",slopes)
    def _bias(self,S,dev):
        pos=torch.arange(S,device=dev).float(); dist=(pos.unsqueeze(0)-pos.unsqueeze(1)).abs()
        return (-self.slopes.view(-1,1,1)*dist.unsqueeze(0)
                +torch.triu(torch.full((S,S),float("-inf"),device=dev),diagonal=1).unsqueeze(0))
    def forward(self, x):
        B,S,_=x.shape; res=x; h=self.norm(x)
        QKV=self.qkv(h).reshape(B,S,3,self.n_heads,self.d_head); Q,K,V=QKV.unbind(2)
        Q=Q.transpose(1,2); K=K.transpose(1,2); V=V.transpose(1,2)
        attn=torch.softmax(torch.matmul(Q,K.transpose(-2,-1))*self.scale+self._bias(S,x.device),dim=-1).nan_to_num(0.)
        return res+self.drop(self.proj(self.drop(attn).matmul(V).transpose(1,2).reshape(B,S,-1)))

class WindowEncoder(nn.Module):
    def __init__(self, in_channels=N_CHAN, d_model=D_MODEL):
        super().__init__()
        g=min(8,d_model//6)
        self.conv1=nn.Conv1d(in_channels,d_model,7,padding=3); self.norm1=nn.GroupNorm(g,d_model)
        self.conv2=nn.Conv1d(d_model,d_model,5,padding=2);     self.norm2=nn.GroupNorm(g,d_model)
        self.conv3=nn.Conv1d(d_model,d_model,3,padding=1);     self.norm3=nn.GroupNorm(g,d_model)
        self.drop=nn.Dropout(0.1); self.cls=nn.Parameter(torch.randn(1,1,d_model)*0.02)
        self.pool=nn.MultiheadAttention(d_model,num_heads=4,dropout=0.1,batch_first=True)
    def forward(self, x):
        x=F.gelu(self.norm1(self.conv1(x))); x=F.gelu(self.norm2(self.conv2(x)))
        x=F.gelu(self.norm3(self.conv3(x))); x=self.drop(x.transpose(1,2))
        out,_=self.pool(self.cls.expand(x.size(0),-1,-1),x,x); return out.squeeze(1)

class VelocityHead(nn.Module):
    def __init__(self, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        h=d_model//2
        def _b(): return nn.Sequential(nn.LayerNorm(d_model),nn.Linear(d_model,h),
                                        nn.GELU(),nn.Dropout(dropout),nn.Linear(h,3))
        self.head_aided=_b(); self.head_dr=_b(); self.v_prev_proj=nn.Linear(3,d_model)
    def forward(self, tokens, outage_flag, v_prev):
        alpha=outage_flag.unsqueeze(-1)
        return (1.-alpha)*self.head_aided(tokens)+alpha*self.head_dr(tokens+alpha*self.v_prev_proj(v_prev))

class MARSNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.window_enc=WindowEncoder(N_CHAN,D_MODEL); self.pos_enc=OutageStepPE(D_MODEL)
        self.tcn=TCNBackbone(D_MODEL,N_TCN_STACKS); self.attn=ALiBiCausalAttention(D_MODEL,N_HEADS,DROPOUT)
        self.head=VelocityHead(D_MODEL,DROPOUT)
    def forward(self, x, outage_flag, v_prev):
        B,S,W,C=x.shape
        tokens=self.window_enc(x.reshape(B*S,W,C).permute(0,2,1).contiguous()).view(B,S,-1)
        tokens=self.pos_enc(tokens,outage_flag); tokens=self.tcn(tokens); tokens=self.attn(tokens)
        return self.head(tokens,outage_flag,v_prev)

print("MARSNet (R20 — architecture unchanged from R14) defined.")

In [ ]:
def combined_loss_v10_batched(dv_pred_norm, dv_true_norm,
                               outage_mask, dv_iqr_t, dv_median_t,
                               v_prev=None, x_raw=None,
                               lam_p=0.0, lam_d=0.0, cap_m=100.0):
    """
    R20 loss — key change: L_cvprior (gated constant-velocity prior).

    L_cvprior vs R19's L_vprev:
      R19: L_vprev fired on ALL outage windows → competed with L_dr on turns → overwhelmed
      R20: L_cvprior fires ONLY when |gyro_z| < CVPRIOR_GYRO_THR=0.10 rad/s
           On straight windows: L_dr≈0, L_cvprior=0.50 → wins cleanly → predict v_prev
           On turn windows:     L_dr=0.90, L_cvprior=0.00 → L_dr teaches turns

    Physics: |gyro_z| < 0.10 rad/s → coordinated straight cruise → v ≈ constant = v_prev
    This is the yaw rate gate. Do NOT use accel_y (vibration contaminates it in this dataset).
    """
    B,S,_=dv_pred_norm.shape; device=dv_pred_norm.device
    iqr=dv_iqr_t.to(device); med=dv_median_t.to(device); aw=DR_AXIS_WEIGHTS.to(device)
    out_mask=outage_mask>0.5; aid_mask=~out_mask

    # ── L_data: GPS-aided windows, xy only ───────────────────────────────────
    L_data=(F.huber_loss(dv_pred_norm[aid_mask][:,:2],dv_true_norm[aid_mask][:,:2],
                         delta=HUBER_DELTA,reduction='mean')
            if aid_mask.any() else torch.tensor(0.,device=device))

    # ── L_dr: all outage windows, axis-weighted ───────────────────────────────
    if out_mask.any():
        L_dr=F.huber_loss(dv_pred_norm[out_mask]*aw,dv_true_norm[out_mask]*aw,
                          delta=HUBER_DELTA,reduction='mean')
    else:
        L_dr=torch.tensor(0.,device=device)

    # ── L_cvprior: GATED constant-velocity prior ─────────────────────────────
    # Gate: |gyro_z| < CVPRIOR_GYRO_THR AND window is in outage
    # gyro_z = channel 5 (yaw rate in body frame)
    # Use midpoint sample of the window (sample WIN_LEN//2)
    L_cvprior=torch.tensor(0.,device=device)
    n_straight_out=0  # track for diagnostics
    if x_raw is not None and out_mask.any() and v_prev is not None:
        gyro_z=x_raw[:,:,WIN_LEN//2,5].abs()             # (B,S) yaw rate magnitude
        straight_out=(gyro_z<CVPRIOR_GYRO_THR)&out_mask  # (B,S) bool
        n_straight_out=straight_out.sum().item()
        if straight_out.any():
            # v3.1: target is zero — in Δv space, straight cruise = no velocity change.
            # This generalises across all flight speeds: Δv=0 regardless of absolute speed.
            zero_target = torch.zeros_like(dv_pred_norm[straight_out][:, :2])
            L_cvprior=F.huber_loss(
                dv_pred_norm[straight_out][:,:2],
                zero_target,
                delta=HUBER_DELTA,reduction='mean')

    # ── L_smooth: jerk penalty ────────────────────────────────────────────────
    L_smooth=(dv_pred_norm[:,1:,:]-dv_pred_norm[:,:-1,:]).pow(2).mean()

    # ── L_phys: ZUPT gate ─────────────────────────────────────────────────────
    L_phys=torch.tensor(0.,device=device)
    if lam_p>0 and x_raw is not None:
        gm=x_raw[:,:,WIN_LEN//2,3:6].norm(dim=-1)
        zm=(gm<ZUPT_GYRO_THR)&out_mask
        if zm.any(): L_phys=(dv_pred_norm[zm].norm(dim=-1)*gm[zm]).mean()

    # ── L_trans: boundary spike ───────────────────────────────────────────────
    trans_list=[]
    for b in range(B):
        oi=out_mask[b].nonzero(as_tuple=True)[0]
        if len(oi)>0: trans_list.append(dv_pred_norm[b,oi[0]].pow(2).mean())
    L_trans=torch.stack(trans_list).mean() if trans_list else torch.tensor(0.,device=device)

    # ── L_drift: cumulative position error ───────────────────────────────────
    L_drift=torch.tensor(0.,device=device)
    if lam_d>0 and out_mask.any():
        dl=[]
        for b in range(B):
            oi=out_mask[b].nonzero(as_tuple=True)[0]
            if len(oi)==0: continue
            pm=dv_pred_norm[b,oi]*iqr+med; tm=dv_true_norm[b,oi]*iqr+med
            pe=((pm-tm)*DT).cumsum(0).norm(dim=-1)
            dl.append(pe[-1].clamp(max=cap_m))
        if dl: L_drift=torch.stack(dl).mean()

    total=(L_data+LAM_DR*L_dr+LAM_CVPRIOR*L_cvprior
           +LAM_SMOOTH*L_smooth+lam_p*L_phys+LAM_DRIFT*lam_d*L_drift+0.05*L_trans)

    return total,{'L_data':L_data.item(),'L_dr':L_dr.item(),'L_cvprior':L_cvprior.item(),
                  'L_phys':L_phys.item(),'L_smooth':L_smooth.item(),
                  'L_drift':L_drift.item(),'L_trans':L_trans.item(),
                  'L_heading':0.0,'n_straight_out':float(n_straight_out)}

print("Loss v10 (gated L_cvprior) defined.")
print(f"  Gate: |gyro_z| < {CVPRIOR_GYRO_THR} rad/s AND outage → L_cvprior fires")
print(f"  Weight: LAM_CVPRIOR={LAM_CVPRIOR}  (safe — no competing L_dr on straight windows)")
print(f"  Target: v_pred → v_prev  (not zero — zero = Y_median = 0.82 m/s, wrong!)")

In [ ]:
import sys, os as _os
sys.path.insert(0, _os.path.abspath(_os.path.join('..', 'data')))
from data_loader import MARSDataset

_npz=np.load(DATA_PATH)
dv_iqr=_npz['Y_iqr'].astype(np.float32); dv_median=_npz['Y_median'].astype(np.float32)
dv_iqr_t=torch.from_numpy(dv_iqr).to(DEVICE); dv_med_t=torch.from_numpy(dv_median).to(DEVICE)
DV_IQR_TRUE=np.array([0.055,0.319,0.00078],dtype=np.float32)
DV_IQR_TRUE_T=torch.from_numpy(DV_IQR_TRUE).to(DEVICE)
print(f"Y_iqr={dv_iqr}  Y_median={dv_median}")
print(f"NOTE: model output is GPS velocity in Y_iqr space. v_pred=0 → v_y={dv_median[1]:.3f} m/s physical!")

train_ds=MARSDataset(DATA_PATH,split='train',outage_prob=0.8)
val_ds=MARSDataset(DATA_PATH,split='val',outage_prob=0.0)
X_tr=_npz['X_train']; W_train_flat=_npz['W_train']; train_valid_idx=_npz['train_valid_idx']
N_SEQS=len(train_valid_idx)

# WeightedRandomSampler — indexed by sequence, NOT flat window index
sample_weights=np.zeros(N_SEQS,dtype=np.float32)
for i,start in enumerate(train_valid_idx):
    end=min(int(start)+SEQ_LEN,len(X_tr))
    gyro=np.linalg.norm(X_tr[int(start):end,WIN_LEN//2,3:6],axis=-1)
    sample_weights[i]=(W_TURN if np.any(gyro>TURN_GYRO_THR)
                       else float(np.mean(W_train_flat[int(start):end])))
sample_weights/=sample_weights.sum()

sampler=WeightedRandomSampler(torch.from_numpy(sample_weights).float(),num_samples=N_SEQS,replacement=True)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,sampler=sampler,num_workers=0,pin_memory=(DEVICE.type=='cuda'))
val_loader=DataLoader(val_ds,batch_size=8,shuffle=False,num_workers=0)
n_batches=math.ceil(N_SEQS/BATCH_SIZE)
print(f"Train seqs:{N_SEQS}  Val seqs:{len(val_ds)}  Batches/epoch:{n_batches}")

In [ ]:
# ── v3.1 Sanity Check: Δv normalisation scale ─────────────────────────────
print('=== v3.1 PRE-TRAINING SANITY CHECK ===')
print(f'Y_iqr (Δv scale):    {dv_iqr}')
print(f'Y_median (Δv):       {dv_median}')
print(f'Expected: Y_iqr ≈ [0.1, 0.1, 0.1], Y_median ≈ 0')
print()

# Check a few training sequences: what is the actual Δv during outage?
_npz = np.load(DATA_PATH)
Ytr  = _npz['Y_train'].astype(np.float32)
tri  = _npz['train_valid_idx']
print('Sample Δv magnitudes during outage (first 5 train sequences):')
for si in range(5):
    start = int(tri[si])
    yrs   = Ytr[start:start+SEQ_LEN]   # absolute GPS vel
    os_   = SEQ_LEN // 3
    # v_prev = GPS vel at window before outage
    vprev = yrs[os_-1]
    # Δv during outage = GPS vel - v_prev
    dv_outage = yrs[os_:os_+100] - vprev
    dv_norm   = dv_outage / (dv_iqr + 1e-8)
    print(f'  S{si}: |Δv_y|_phys={np.abs(dv_outage[:,1]).mean():.4f} m/s  '
          f'|Δv_y|_norm={np.abs(dv_norm[:,1]).mean():.4f}  '
          f'(scale=Y_iqr[1]={dv_iqr[1]:.3f})')

print()
# Check loss balance at epoch 0 will be ok
# L_dr weight 0.90, L_cvprior weight LAM_CVPRIOR
# If Δv_norm magnitudes are ~0-5, loss will be ~0.1-2.0 Huber units
# L_cvprior should be smaller than L_dr initially
print(f'LAM_CVPRIOR: {LAM_CVPRIOR}  (consider 0.10-0.20 if Δv_norm > 2.0)')
print('=== END SANITY CHECK ===')

In [ ]:
def compute_v_prev(y_norm, outage_mask, dv_iqr_t, dv_med_t):
    device=y_norm.device; B,S,_=y_norm.shape
    v_raw=y_norm*dv_iqr_t+dv_med_t
    not_flag=~(outage_mask>0.5)
    t_idx=torch.arange(S,device=device).unsqueeze(0).expand(B,-1)
    gps_idx=torch.where(not_flag,t_idx,torch.zeros_like(t_idx))
    last_gps=gps_idx.cummax(dim=1).values
    v_prev=torch.stack([v_raw[:,:,ax].gather(1,last_gps) for ax in range(3)],dim=-1)
    return v_prev,v_raw

print("compute_v_prev (vectorised) defined.")

In [ ]:
torch.manual_seed(42)
model=MARSNet().to(DEVICE)
n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')
n_batches=math.ceil(N_SEQS/BATCH_SIZE)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.OneCycleLR(
    optimizer,max_lr=LR,total_steps=MAX_EPOCHS*n_batches,
    pct_start=0.05,anneal_strategy='cos',div_factor=25,final_div_factor=100)

# Speed benchmark
model.eval()
_b=next(iter(train_loader)); _xn=_b['x_norm'].to(DEVICE); _yn=_b['y_norm'].to(DEVICE); _om=_b['outage_mask'].to(DEVICE)
B,S,W,_=_xn.shape; _vp=torch.zeros(B,S,3,device=DEVICE); _flag=_om.float()
_vpch=_vp.unsqueeze(2).expand(-1,-1,W,-1); _fch=_flag.unsqueeze(-1).unsqueeze(-1).expand(-1,-1,W,1)
_xf=torch.cat([_xn[:,:,:,:N_IMU_CHAN],_vpch,_fch],dim=-1)
with torch.no_grad():
    for _ in range(3): model(_xf,_flag,_vp)
if DEVICE.type=='cuda': torch.cuda.synchronize()
t0=time.time()
with torch.no_grad():
    for _ in range(10): model(_xf,_flag,_vp)
if DEVICE.type=='cuda': torch.cuda.synchronize()
ms=(time.time()-t0)*100
print(f"Forward pass:{ms:.1f}ms  Est epoch:{ms/1000*n_batches*2.2:.0f}s")

In [ ]:
print("="*65); print("  RUN 20 PRE-TRAINING SANITY CHECKS"); print("="*65)
checks=[]

ok1=(LAM_CVPRIOR==0.50); checks.append(ok1)
print(f"[1] LAM_CVPRIOR={LAM_CVPRIOR}  (should be 0.50)"," ✓" if ok1 else " ✗")

ok2=(CVPRIOR_GYRO_THR==0.10); checks.append(ok2)
print(f"[2] CVPRIOR_GYRO_THR={CVPRIOR_GYRO_THR}  (must be 0.10 rad/s from R19 data)"," ✓" if ok2 else " ✗")

ok3=(LAM_VPREV==0.0); checks.append(ok3)
print(f"[3] LAM_VPREV={LAM_VPREV}  (must be 0.0 — ungated version removed)"," ✓" if ok3 else " ✗")

ok4=(P_VPREV_MASK==0.0); checks.append(ok4)
print(f"[4] P_VPREV_MASK={P_VPREV_MASK}  (must be 0.0 — caused R17 catastrophe)"," ✓" if ok4 else " ✗")

ok5=(DRIFT_WARM_START==30); checks.append(ok5)
print(f"[5] DRIFT_WARM_START={DRIFT_WARM_START}  (should be 30)"," ✓" if ok5 else " ✗")

ok6=(TURN_GYRO_THR==0.10); checks.append(ok6)
print(f"[6] TURN_GYRO_THR={TURN_GYRO_THR}  (corrected: R14-R19 used 0.50 which was 5σ above mean)"," ✓" if ok6 else " ✗")

# Run one batch through v10 loss
model.eval()
batch=next(iter(train_loader))
xn=batch['x_norm'].to(DEVICE); yn=batch['y_norm'].to(DEVICE)
om=batch['outage_mask'].to(DEVICE); xr=batch['x_raw'].to(DEVICE)
B,S,W,_=xn.shape
vp,_=compute_v_prev(yn,om,dv_iqr_t,dv_med_t)
flag=om.float(); xi=xn[:,:,:,:N_IMU_CHAN]
vpch=vp.unsqueeze(2).expand(-1,-1,W,-1); fch=flag.unsqueeze(-1).unsqueeze(-1).expand(-1,-1,W,1)
xf=torch.cat([xi,vpch,fch],dim=-1)
with torch.no_grad():
    dvp=model(xf,flag,vp)
    loss0,comps=combined_loss_v10_batched(dvp,yn,om,dv_iqr_t,dv_med_t,v_prev=vp,x_raw=xr)

print(f"\n[7] Epoch-0 loss components (v10):")
for k,v in comps.items():
    if k=='n_straight_out':
        pct=100*v/(om>0.5).sum().item() if (om>0.5).sum().item()>0 else 0
        note=f"  ({pct:.0f}% of outage windows — expect ~87%)"
        print(f"    {'n_straight_out':<18} = {v:.0f}{note}"); continue
    note=""
    if k=='L_cvprior' and v<1e-4: note="  ✗ NOT FIRING — check x_raw passed to loss"
    if k=='L_cvprior' and v>1e-4: note="  ✓ firing"
    if k=='L_heading' and v!=0:   note="  ✗ MUST BE 0"
    print(f"    {k:<18} = {v:.5f}{note}")

ok7=(comps.get('L_cvprior',0)>1e-4 and comps.get('L_heading',0)==0)
checks.append(ok7)
n_out=comps.get('n_straight_out',0); n_total=(om>0.5).sum().item()
pct_gate=100*n_out/max(n_total,1)
print(f"    L_cvprior firing:{comps.get('L_cvprior',0)>1e-4}  gate coverage:{pct_gate:.0f}%"," ✓" if ok7 else " ✗")

# Check gate not 100% (would mean all windows gated = bad)
ok8=(40<pct_gate<98); checks.append(ok8)
print(f"[8] Gate coverage {pct_gate:.0f}% (should be 40–98%, expect ~87%)"," ✓" if ok8 else " ✗ unexpected coverage")

ok9=not torch.isnan(dvp).any().item(); checks.append(ok9)
print(f"[9] NaN in output: {not ok9}"," ✓ clean" if ok9 else " ✗")

# L_dr must dominate over LAM_CVPRIOR*L_cvprior on gated windows
# weighted: 0.90*L_dr vs 0.50*L_cvprior
wdr=LAM_DR*comps.get('L_dr',0); wcp=LAM_CVPRIOR*comps.get('L_cvprior',0)
ok10=(wdr>=wcp*0.5); checks.append(ok10)  # loose — ok if roughly similar at init
print(f"[10] Weighted L_dr={wdr:.4f} vs LAM_CVPRIOR*L_cvprior={wcp:.4f}  ratio={wdr/(wcp+1e-8):.2f}x"," ✓" if ok10 else " ⚠ check")

print("\n"+"="*65)
n_pass=sum(checks)
print(f"  {n_pass}/{len(checks)} checks passed.",
      "✓ Proceed to training." if n_pass>=len(checks)-1 else "✗ Fix failures first.")
print("="*65)

In [ ]:
def train_one_epoch(model,loader,optimizer,scheduler,dv_iqr_t,dv_median_t,device,lam_p=0.0,lam_d=0.0):
    model.train(); totals=defaultdict(float); n=0
    for batch in loader:
        xn=batch['x_norm'].to(device); yn=batch['y_norm'].to(device)
        om=batch['outage_mask'].to(device); xr=batch['x_raw'].to(device)
        B,S,W,_=xn.shape
        vp,_=compute_v_prev(yn,om,dv_iqr_t.to(device),dv_median_t.to(device))
        flag=om.float(); xi=xn[:,:,:,:N_IMU_CHAN]
        if TEMPORAL_DROPOUT_P>0:
            xi=xi*(torch.rand(B,S,1,1,device=device)>TEMPORAL_DROPOUT_P).float()
        vpch=vp.unsqueeze(2).expand(-1,-1,W,-1); fch=flag.unsqueeze(-1).unsqueeze(-1).expand(-1,-1,W,1)
        xf=torch.cat([xi,vpch,fch],dim=-1)
        optimizer.zero_grad()
        dv_pred=model(xf,flag,vp)
        loss,comps=combined_loss_v10_batched(dv_pred,yn,om,dv_iqr_t,dv_median_t,
                                              v_prev=vp,x_raw=xr,lam_p=lam_p,lam_d=lam_d)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        totals['total']+=loss.item()
        for k,v in comps.items(): totals[k]+=v
        n+=1
    return {k:v/max(n,1) for k,v in totals.items()}

print("train_one_epoch (R20 — v10 loss, gated L_cvprior) defined.")

In [ ]:
history={k:[] for k in ['train_loss','val_drift_m','turn_rmse','straight_rmse',
    'L_data','L_dr','L_cvprior','L_phys','L_smooth','L_drift','L_heading',
    'sr_x','sr_y','L_cvprior_val','mean_vprev_err_ms','gate_pct']}
best_drift=float('inf'); best_state=None; patience_ctr=0; start_epoch=1

print(f"{'Ep':>4} {'Loss':>8} {'Drift':>8} {'L_cvp':>7} {'VpErr':>7} {'Gate%':>6} {'sr_y':>6} {'t(s)':>6}  Status")
print('-'*80)

for epoch in range(start_epoch,MAX_EPOCHS+1):
    ep_t0=time.time()
    lam_p=min(LAM_PHYS_MAX,LAM_PHYS_MAX*max(0,epoch-PHYS_WARM_START)/max(1,PHYS_WARM_END-PHYS_WARM_START))
    lam_d=min(1.,max(0.,epoch-DRIFT_WARM_START)/max(1,DRIFT_WARM_END-DRIFT_WARM_START))
    tc=train_one_epoch(model,train_loader,optimizer,scheduler,dv_iqr_t,dv_med_t,DEVICE,lam_p,lam_d)

    model.eval()
    val_drifts,sr_xs,sr_ys,turn_e,str_e,vprev_vals,vprev_err,gate_pcts=[],[],[],[],[],[],[],[]
    with torch.no_grad():
        for batch in val_loader:
            xn=batch['x_norm'].to(DEVICE); yn=batch['y_norm'].to(DEVICE)
            om=batch['outage_mask'].to(DEVICE); xr=batch['x_raw'].to(DEVICE)
            B,S,W,_=xn.shape
            vp,_=compute_v_prev(yn,om,dv_iqr_t,dv_med_t)
            flag=om.float(); xi=xn[:,:,:,:N_IMU_CHAN]
            vpch=vp.unsqueeze(2).expand(-1,-1,W,-1); fch=flag.unsqueeze(-1).unsqueeze(-1).expand(-1,-1,W,1)
            xf=torch.cat([xi,vpch,fch],dim=-1); dvp=model(xf,flag,vp)

            # Gate coverage on val
            omm=om>0.5; gz=xr[:,:,WIN_LEN//2,5].abs()
            str_out=(gz<CVPRIOR_GYRO_THR)&omm
            if omm.any(): gate_pcts.append(100*str_out.sum().item()/omm.sum().item())

            # L_cvprior on val
            if str_out.any() and vp is not None:
                # v3.1: L_cvprior targets zero, so VpErr is |dv_pred| on straight windows
                zero_t = torch.zeros_like(dvp[str_out][:,:2])
                lv=F.huber_loss(dvp[str_out][:,:2],zero_t,delta=HUBER_DELTA,reduction='mean').item()
                vprev_vals.append(lv)
                vprev_err.append(dvp[str_out][:,1].abs().mean().item()*dv_iqr_t[1].item())

            for b in range(B):
                oi=om[b].nonzero(as_tuple=True)[0]
                if len(oi)==0: continue
                # v3.1: dv is Δv. Position integrates v_prev+Δv as absolute velocity.
                vp_b   = vp[b]                        # (S,3) absolute GPS vel
                pm_dv  = dvp[b]*dv_iqr_t+dv_med_t    # predicted Δv physical
                tm_dv  = yn[b]*dv_iqr_t+dv_med_t     # true Δv physical
                pm     = vp_b + pm_dv                 # predicted absolute vel
                tm = vp_b + tm_dv                 # true absolute vel
                pe=((pm-tm)*DT)[oi].cumsum(0).norm(dim=-1); val_drifts.append(pe[-1].item())
                ps=dvp[b,oi].std(dim=0); ts=yn[b,oi].std(dim=0)
                sr_xs.append((ps[0]/(ts[0]+1e-8)).item()); sr_ys.append((ps[1]/(ts[1]+1e-8)).item())
                gm=xr[b,:,W//2,3:6].norm(dim=-1)
                for idx in oi:
                    e2=(pm[idx]-tm[idx]).pow(2).mean().item()
                    if gm[idx]>TURN_GYRO_THR: turn_e.append(e2)
                    elif gm[idx]>=ZUPT_GYRO_THR: str_e.append(e2)

    vd=float(np.mean(val_drifts)) if val_drifts else float('nan')
    srx=float(np.mean(sr_xs)) if sr_xs else float('nan')
    sry=float(np.mean(sr_ys)) if sr_ys else float('nan')
    trmse=float(np.sqrt(np.mean(turn_e))) if turn_e else float('nan')
    srmse=float(np.sqrt(np.mean(str_e))) if str_e else float('nan')
    lv_v=float(np.mean(vprev_vals)) if vprev_vals else float('nan')
    vp_ms=float(np.mean(vprev_err)) if vprev_err else float('nan')
    gp=float(np.mean(gate_pcts)) if gate_pcts else float('nan')
    ep_s=time.time()-ep_t0

    history['train_loss'].append(tc['total']); history['val_drift_m'].append(vd)
    history['turn_rmse'].append(trmse); history['straight_rmse'].append(srmse)
    history['sr_x'].append(srx); history['sr_y'].append(sry)
    history['L_cvprior_val'].append(lv_v); history['mean_vprev_err_ms'].append(vp_ms)
    history['gate_pct'].append(gp)
    for k in ['L_data','L_dr','L_cvprior','L_phys','L_smooth','L_drift','L_heading']:
        history[k].append(tc.get(k,0.))

    is_best=(not math.isnan(vd)) and (vd<best_drift)
    if is_best:
        best_drift=vd; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        patience_ctr=0; status=f'★ BEST {best_drift:.2f}m'
    else:
        patience_ctr+=1; status=f'({patience_ctr}/{PATIENCE})'

    torch.save({'model':model.state_dict(),'epoch':epoch,'best_val':best_drift,
                'history':history,'optimizer':optimizer.state_dict(),
                'scheduler':scheduler.state_dict(),'patience_ctr':patience_ctr,
                'DV_iqr':dv_iqr,'DV_median':dv_median,'DV_IQR_TRUE':DV_IQR_TRUE,'run':'20'},CKPT_LAST)

    if epoch==1 or epoch%5==0 or is_best:
        lcvp=tc.get('L_cvprior',0.)
        print(f'{epoch:>4} {tc["total"]:>8.4f} {vd:>8.2f}m {lcvp:>7.4f} {vp_ms:>7.3f} '
              f'{gp:>5.1f}% {sry:>6.2f} {ep_s:>6.0f}  {status}')
        if epoch==1 and lcvp<0.001: print("  ✗ L_cvprior not firing — check x_raw is passed")
        if epoch==1 and vd>80: print("  ⚠ High ep1 drift — normal if gate working; watch ep5")
        if epoch==30 and vp_ms>0.5: print(f"  ⚠ VpErr={vp_ms:.3f} — target <0.5 by ep30. If stuck, try LAM_CVPRIOR=1.0")
        if epoch==30 and vd>8: print(f"  ⚠ drift={vd:.1f}m — target <6m. Check S47-S52 group specifically")
        if epoch==60 and vd>5: print(f"  ⚠ drift={vd:.1f}m — target <4m at ep60 with drift loss active")

    if patience_ctr>=PATIENCE:
        print(f'Early stop at epoch {epoch}'); break

print(f'\nBest val drift:{best_drift:.3f}m  (R16 benchmark:6.80m  Naive:5.96m)')
if best_drift<5.0: print("✓ Paper target met — MARSNet beats naive baseline!")
elif best_drift<6.8: print("~ Close to R16. Run full eval and compare groups.")
else: print("✗ Still above R16. Check S47-S52 in full eval.")

In [ ]:
if best_state is not None:
    torch.save({'model':best_state,'epoch':epoch,'best_val':best_drift,
                'DV_iqr':dv_iqr,'DV_median':dv_median,'DV_IQR_TRUE':DV_IQR_TRUE,
                'history':history,'run':'20'},CKPT_PATH)
    print(f'Best → {CKPT_PATH}'); print(f'Last → {CKPT_LAST}')

In [ ]:
def save_training_history(history,save_path=None):
    fig,axes=plt.subplots(1,4,figsize=(22,5))
    axes[0].plot(history.get('train_loss',[])); axes[0].set_title('Train Loss')
    axes[1].plot(history.get('val_drift_m',[])); axes[1].set_title('Val Drift (m)')
    axes[1].axhline(5.0,color='g',ls='--',lw=1,label='5m target'); axes[1].legend(fontsize=8)
    axes[2].plot(history.get('L_cvprior',[]),label='train')
    axes[2].plot(history.get('L_cvprior_val',[]),label='val',ls='--')
    axes[2].axhline(0.01,color='g',ls='--',lw=1,label='target<0.01'); axes[2].set_title('L_cvprior'); axes[2].legend(fontsize=8)
    axes[3].plot(history.get('mean_vprev_err_ms',[])); axes[3].axhline(0.05,color='g',ls='--',lw=1,label='target')
    axes[3].axhline(0.5,color='orange',ls=':',lw=1); axes[3].set_title('|v_pred_y−v_prev_y| m/s'); axes[3].legend(fontsize=8)
    plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=120,bbox_inches='tight')
    plt.show(); plt.close()

def save_val_metrics(history,save_path=None):
    keys=[('val_drift_m','Val Drift (m)'),('turn_rmse','Turn RMSE'),
          ('straight_rmse','Straight RMSE'),('train_loss','Train Loss'),
          ('sr_x','SR-x'),('sr_y','SR-y'),('L_cvprior','L_cvprior'),('gate_pct','Gate % (straight out)')]
    fig,axes=plt.subplots(2,4,figsize=(20,8)); axes=axes.flatten()
    for ax,(k,lbl) in zip(axes,keys):
        d=history.get(k,[])
        if d: ax.plot(d)
        if k=='gate_pct': ax.axhline(87,color='g',ls='--',lw=1,label='expected 87%'); ax.legend(fontsize=7)
        if k in ('sr_x','sr_y'):
            ax.axhline(1.,color='g',ls='--',lw=1); ax.axhline(3.,color='r',ls=':',lw=1)
        ax.set_title(lbl); ax.grid(True,alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=120,bbox_inches='tight')
    plt.show(); plt.close()

save_training_history(history,os.path.join(PLOT_DIR,'training_history.png'))
save_val_metrics(history,os.path.join(PLOT_DIR,'val_metrics.png'))
print(f'Plots → {PLOT_DIR}/')

In [ ]:
@torch.no_grad()
def predict_sequence(model,x_norm_seq,y_raw_seq,outage_start,outage_end,dv_iqr,dv_median,device):
    model.eval(); S=x_norm_seq.shape[0]; W=x_norm_seq.shape[1]
    om=torch.zeros(S,dtype=torch.float32)
    if outage_start>=3: om[outage_start-2]=1/3; om[outage_start-1]=2/3
    om[outage_start:outage_end]=1.
    vr=torch.from_numpy(y_raw_seq).float(); vp=torch.zeros(S,3); last=vr[0].clone()
    for t in range(S):
        if om[t]>0.5: vp[t]=last
        else: vp[t]=vr[t]; last=vr[t].clone()
    xi=torch.from_numpy(x_norm_seq).float()[:,:,:N_IMU_CHAN]
    vpch=vp.unsqueeze(1).expand(-1,W,-1); fch=om.view(S,1,1).expand(-1,W,1)
    xf=torch.cat([xi,vpch,fch],dim=-1)
    dvp=model(xf.unsqueeze(0).to(device),om.unsqueeze(0).to(device),vp.unsqueeze(0).to(device)).squeeze(0).cpu()
    iqr=torch.from_numpy(dv_iqr).float(); med=torch.from_numpy(dv_median).float()
    pm=dvp*iqr+med  # predicted Δv in physical units
    tm=vr           # y_raw_seq = absolute GPS vel (used as v_prev and true ref)
    # v3.1: position integrates (v_prev + Δv_pred) as instantaneous velocity
    # vp[t] = last known absolute GPS vel before/during outage
    # True absolute vel at t = vp[t] + Δv_true[t] = vr[t] (GPS vel)
    # Pred absolute vel at t = vp[t] + pm[t]
    def integrate(dv_delta, v_prev_phys, s):
        pos=torch.zeros(S,2)
        for t in range(s+1,S):
            v_abs = v_prev_phys[t,:2] + dv_delta[t,:2]  # absolute velocity
            pos[t]=pos[t-1]+v_abs*DT
        return pos.numpy()
    pp=integrate(pm, vp, outage_start)   # predicted path
    pt=integrate(torch.zeros_like(pm), vp, outage_start)  # true path (Δv=0 → use v_prev=GPS vel)
    # For true path: vp[t]=GPS vel during aided, GPS vel at outage_start during outage
    # But we want true path from actual GPS → use vr directly
    def integrate_true(v_abs, s):
        pos=torch.zeros(S,2)
        for t in range(s+1,S): pos[t]=pos[t-1]+v_abs[t,:2]*DT
        return pos.numpy()
    pt=integrate_true(vr, outage_start)  # true path from actual GPS vel
    return {'dv_pred_ms':pm.numpy(),'dv_true_ms':tm.numpy(),'pos_pred':pp,'pos_true':pt,
            'outage_start':outage_start,'drift_m':float(np.linalg.norm(pp[outage_end-1]-pt[outage_end-1]))}

def plot_path_2d_grid(eval_results,save_path,n_cols=6):
    n=len(eval_results); nr=math.ceil(n/n_cols)
    fig,axes=plt.subplots(nr,n_cols,figsize=(n_cols*3.2,nr*2.8),squeeze=False)
    for i,res in enumerate(eval_results):
        ax=axes[i//n_cols][i%n_cols]; gt=res['pos_gt']; pred=res['pos_pred']; oi=res['outage_start']
        ax.plot(gt[:oi,1],gt[:oi,0],'b-',lw=1.); ax.plot(gt[oi:,1],gt[oi:,0],'b--',lw=1.,alpha=0.5)
        ax.plot(pred[oi:,1],pred[oi:,0],'r-',lw=1.); ax.plot(gt[oi,1],gt[oi,0],'ko',ms=3,zorder=5)
        d=res['drift_m']; c='green' if d<5 else('darkorange' if d<15 else 'red')
        beats=res.get('naive_drift_m',999)>d
        ax.set_title(f"S{res['seq_idx']} {'✓' if beats else '✗'} {d:.1f}m",fontsize=7.5,color=c,fontweight='bold')
        ax.set_aspect('equal'); ax.tick_params(labelsize=5); ax.grid(True,alpha=0.25)
    for j in range(n,nr*n_cols): axes[j//n_cols][j%n_cols].set_visible(False)
    fig.suptitle('Run 20 — All Val Seqs | Green<5m  Orange<15m  Red≥15m',fontsize=10)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.show(); plt.close()

print("predict_sequence + grid plot defined.")

In [ ]:
if best_state is not None:
    model.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})

_nv=np.load(DATA_PATH); Xv=_nv['X_val']; Yv=_nv['Y_val']
Xmed=_nv['X_median']; Xiq=_nv['X_iqr']; vi=_nv['val_valid_idx']
SL=int(_nv['seq_len'][0]) if 'seq_len' in _nv else SEQ_LEN

GROUP={**{i:'straight-short' for i in range(0,35)},**{i:'straight-med' for i in range(35,41)},
       **{i:'TURN' for i in range(41,47)},**{i:'FALSE-ALARM' for i in range(47,53)},
       **{i:'long-outage' for i in range(53,59)}}

all_res=[]; mdrifts=[]; ndrifts=[]
print(f"{'S':>3}  {'Model':>7}  {'Naive':>7}  {'Beats?':>7}  Group"); print("-"*50)
for si,start in enumerate(vi):
    xrs=Xv[int(start):int(start)+SL]; yrs=Yv[int(start):int(start)+SL]
    if len(xrs)<SL: continue
    xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq)
    os_=SL//3; oe_=min(os_+100,SL)
    r=predict_sequence(model,xns,yrs,os_,oe_,dv_iqr,dv_median,DEVICE)
    lv=yrs[os_-1] if os_>0 else np.zeros(3); pn=np.zeros((SL,2))
    for t in range(os_+1,SL): pn[t]=pn[t-1]+lv[:2]*DT
    nd=float(np.linalg.norm(pn[oe_-1]-r['pos_true'][oe_-1]))
    mdrifts.append(r['drift_m']); ndrifts.append(nd)
    all_res.append({'seq_idx':si,'pos_gt':np.column_stack([r['pos_true'],np.zeros(SL)]),
                    'pos_pred':np.column_stack([r['pos_pred'],np.zeros(SL)]),
                    'drift_m':r['drift_m'],'naive_drift_m':nd,'outage_start':os_,'group':GROUP.get(si,'?')})
    print(f"{si:>3}  {r['drift_m']:>6.2f}m  {nd:>6.2f}m  {'BEATS' if r['drift_m']<nd else 'worse':>7}  {GROUP.get(si,'?')}")

ma=np.array(mdrifts); na=np.array(ndrifts)
print("\n"+"="*58)
print(f"  Seqs:{len(ma)}  Mean:{ma.mean():.2f}m  Median:{np.median(ma):.2f}m  90th:{np.percentile(ma,90):.2f}m")
print(f"  % beats naive:{100*np.mean(ma<na):.1f}%  % under 5m:{100*np.mean(ma<5):.1f}%")
print(f"\n  ── R19 benchmarks ──")
print(f"  R19: mean=8.79m  median=2.64m  90th=25.95m  59.3% under 5m")
print(f"  R16: mean=6.80m  median=2.54m  90th=22.53m  69.5% under 5m  [prev best]")
print(f"  Naive: mean=5.96m")
print(f"\n  ── Group breakdown ──")
for grp in ['straight-short','straight-med','TURN','FALSE-ALARM','long-outage']:
    idx=[i for i,r in enumerate(all_res) if r['group']==grp]
    if not idx: continue
    gm=np.array([mdrifts[i] for i in idx]); gn=np.array([ndrifts[i] for i in idx])
    print(f"  {grp:<16}: model={gm.mean():.2f}m  naive={gn.mean():.2f}m  beats={np.mean(gm<gn)*100:.0f}%")
print(f"\n  ── FALSE-ALARM (S47–S52) — key R20 test ──")
r19_fa=[22.54,30.19,33.45,31.55,25.59,25.60]
for i,r in enumerate([r for r in all_res if r['group']=='FALSE-ALARM']):
    verdict='✓ FIXED' if r['drift_m']<3 else('~ improved' if r['drift_m']<15 else '✗ still bad')
    print(f"  S{r['seq_idx']}: {r['drift_m']:.2f}m  naive={r['naive_drift_m']:.2f}m  R19={r19_fa[i]:.1f}m  {verdict}")
print("="*58)

with open(os.path.join(PLOT_DIR,'val_seq_results.csv'),'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['seq_idx','group','model_drift_m','naive_drift_m','beats_naive'])
    w.writeheader()
    for r in all_res:
        w.writerow({'seq_idx':r['seq_idx'],'group':r['group'],
                    'model_drift_m':round(r['drift_m'],4),'naive_drift_m':round(r['naive_drift_m'],4),
                    'beats_naive':r['drift_m']<r['naive_drift_m']})

In [ ]:
plot_path_2d_grid(all_res,os.path.join(PLOT_DIR,'path_grid_all_seqs.png'),n_cols=6)
for res in all_res[:3]:
    sn=res['seq_idx']; st=int(vi[sn]); xrs=Xv[st:st+SL]; yrs=Yv[st:st+SL]
    xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq)
    r=predict_sequence(model,xns,yrs,SL//3,min(SL//3+100,SL),dv_iqr,dv_median,DEVICE)
    oi=res['outage_start']; gt=res['pos_gt']; pred=res['pos_pred']
    fig,ax=plt.subplots(figsize=(9,5))
    ax.plot(gt[:oi,1],gt[:oi,0],'b-',lw=1.5,label='GT (GPS)')
    ax.plot(gt[oi:,1],gt[oi:,0],'b--',lw=1.5,alpha=0.5,label='GT (outage)')
    ax.plot(pred[oi:,1],pred[oi:,0],'r-',lw=1.5,label='Predicted')
    ax.plot(gt[oi,1],gt[oi,0],'ko',ms=6,zorder=5,label='Outage start')
    ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
    ax.set_title(f"Seq {sn} | Model={res['drift_m']:.2f}m  Naive={res['naive_drift_m']:.2f}m")
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3); ax.set_aspect('equal')
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR,f'path_seq{sn}.png'),dpi=150,bbox_inches='tight')
    plt.show(); plt.close()
print("All plots saved.")

In [ ]:
print("="*65); print("  RUN 20 DIAGNOSTICS"); print("="*65)
print("  R19 benchmarks: shuffle=1.04x | TCN=14.53x | VpErr=0.276 m/s | drift=8.79m")

_nv2=np.load(DATA_PATH); Xv2=_nv2['X_val']; Yv2=_nv2['Y_val']; vi2=_nv2['val_valid_idx']
norm_d=[]; norm_preds=[]; norm_trues=[]; norm_vprevs=[]
model.eval()
with torch.no_grad():
    for res in all_res[:10]:
        si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; yrs=Yv2[st:st+SL]
        xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq); os_=SL//3; oe_=min(os_+100,SL)
        r=predict_sequence(model,xns,yrs,os_,oe_,dv_iqr,dv_median,DEVICE)
        norm_d.append(r['drift_m']); norm_preds.append(r['dv_pred_ms'][os_:oe_]); norm_trues.append(r['dv_true_ms'][os_:oe_])
        vr=torch.from_numpy(yrs).float(); vp_=torch.zeros(SL,3); last=vr[0].clone()
        for t in range(SL):
            if t>=os_ and t<oe_: vp_[t]=last
            else:
                if t<os_: vp_[t]=vr[t]; last=vr[t].clone()
        norm_vprevs.append(vp_[os_:oe_].numpy())

# Test 1: v_prev ablation
print("\nTEST 1 — v_prev Ablation  (R19: 1.00x → target >1.5x)")
abl_d=[]
with torch.no_grad():
    for res in all_res[:10]:
        si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; yrs=Yv2[st:st+SL]
        xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq); xns2=xns.copy(); xns2[:,:,10:13]=0.
        r_a=predict_sequence(model,xns2,yrs,SL//3,min(SL//3+100,SL),dv_iqr,dv_median,DEVICE)
        abl_d.append(r_a['drift_m'])
ratio1=np.mean(abl_d)/(np.mean(norm_d)+1e-6)
print(f"  Normal:{np.mean(norm_d):.2f}m  Ablated:{np.mean(abl_d):.2f}m  Ratio:{ratio1:.2f}x")
print(f"  {'✓ v_prev pathway active' if ratio1>1.5 else '~ Marginal' if ratio1>1.1 else '✗ Still coasting'}")

# Test 2: IMU shuffle
print("\nTEST 2 — IMU Temporal Shuffle  (R19: 1.04x)")
shuf_d=[]
with torch.no_grad():
    for res in all_res[:10]:
        si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; yrs=Yv2[st:st+SL]
        xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq); os_=SL//3; oe_=min(os_+100,SL)
        xs=xns.copy(); perm=np.random.permutation(np.arange(os_,oe_)); xs[os_:oe_]=xs[perm]
        r_s=predict_sequence(model,xs,yrs,os_,oe_,dv_iqr,dv_median,DEVICE); shuf_d.append(r_s['drift_m'])
ratio2=np.mean(shuf_d)/(np.mean(norm_d)+1e-6)
print(f"  Normal:{np.mean(norm_d):.2f}m  Shuffled:{np.mean(shuf_d):.2f}m  Ratio:{ratio2:.2f}x")
print(f"  {'✓ Inter-window dependency' if ratio2>1.3 else '~ Per-window features (expected)' if ratio2>0.9 else '✗ Spurious correlation'}")

# Test 3: Per-axis variance & bias
print("\nTEST 3 — Per-Axis Stats  (R19: vy_y ratio=8.82x, |pred-vprev|=0.246)")
pa=np.concatenate(norm_preds); ta=np.concatenate(norm_trues); pv_a=np.concatenate(norm_vprevs)
print(f"  {'Axis':<8} {'RMSE':>8} {'Bias':>9} {'Ratio':>7} {'|pred-vp|':>10}  Status")
print("  "+"-"*55)
for i,ax_n in enumerate(['vy_x','vy_y','vy_z']):
    rmse=np.sqrt(np.mean((pa[:,i]-ta[:,i])**2)); bias=np.mean(pa[:,i]-ta[:,i])
    ts=np.std(ta[:,i]); ps=np.std(pa[:,i]); ratio=ps/(ts+1e-8)
    vpe=np.abs(pa[:,i]-pv_a[:,i]).mean()
    ok=0.5<=ratio<=3.0
    print(f"  {ax_n:<8} {rmse:>8.4f} {bias:>+9.4f} {ratio:>7.2f} {vpe:>10.4f}  {'✓' if ok else '✗'}")
print(f"  Key: |pred-vprev| on vy_y should be <0.05 m/s (R19: 0.246 m/s)")

# Test 4: Turn vs Straight RMSE
print("\nTEST 4 — Turn vs Straight RMSE  (TURN_GYRO_THR now=0.10)")
turn_e2,str_e2=[],[]
for i,res in enumerate(all_res[:10]):
    si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; os_=SL//3; oe_=min(os_+100,SL)
    gyro=np.linalg.norm(xrs[os_:oe_,WIN_LEN//2,3:6],axis=-1)
    p=norm_preds[i]; t_=norm_trues[i]
    for j in range(min(len(gyro),len(p))):
        e2=np.mean((p[j]-t_[j])**2)
        if gyro[j]>TURN_GYRO_THR: turn_e2.append(e2)
        elif gyro[j]>=ZUPT_GYRO_THR: str_e2.append(e2)
if turn_e2 and str_e2:
    tr=np.sqrt(np.mean(turn_e2)); sr=np.sqrt(np.mean(str_e2)); ratio4=tr/(sr+1e-8)
    print(f"  Turn:{tr:.4f}  Straight:{sr:.4f}  Ratio:{ratio4:.3f}x")
    print(f"  {'✓ Regime differentiation' if ratio4>1.3 else '~ Emerging' if ratio4>1.1 else '✗ No differentiation'}")
    print(f"  (Turn windows:{len(turn_e2)}  Straight:{len(str_e2)} — with corrected 0.10 threshold)")
else:
    print(f"  Turn:{len(turn_e2)} windows  Straight:{len(str_e2)}  Insufficient samples")

# Test 5: TCN ablation  (R19: 14.53x — must stay high)
print("\nTEST 5 — TCN Ablation  (R19: 14.53x — must stay high)")
tcn_d=[]; orig={k:v.clone() for k,v in model.state_dict().items()}
with torch.no_grad():
    for name,param in model.named_parameters():
        if 'tcn' in name and 'conv' in name: param.data.zero_()
    for res in all_res[:10]:
        si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; yrs=Yv2[st:st+SL]
        xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq)
        r_t=predict_sequence(model,xns,yrs,SL//3,min(SL//3+100,SL),dv_iqr,dv_median,DEVICE)
        tcn_d.append(r_t['drift_m'])
model.load_state_dict(orig)
ratio5=np.mean(tcn_d)/(np.mean(norm_d)+1e-6)
print(f"  Normal:{np.mean(norm_d):.2f}m  TCN-ablated:{np.mean(tcn_d):.2f}m  Ratio:{ratio5:.2f}x")
print(f"  {'✓ TCN contributing (>1.5x)' if ratio5>1.5 else '✗ TCN degraded — check L_cvprior still active'}")
if ratio5<5: print(f"  WARNING: R19 was 14.53x. If <5x, L_cvprior may not be requiring temporal integration.")

# Test 6: v_prev tracking (key test)
print("\nTEST 6 — v_prev Tracking Accuracy  (R19: 0.276 m/s → target <0.05)")
straight_err,turn_err=[],[]
with torch.no_grad():
    for i,res in enumerate(all_res[:20]):
        si=res['seq_idx']; st=int(vi2[si]); xrs=Xv2[st:st+SL]; yrs=Yv2[st:st+SL]
        xns=(xrs-Xmed)/np.where(Xiq<1e-6,1.,Xiq); os_=SL//3; oe_=min(os_+100,SL)
        r=predict_sequence(model,xns,yrs,os_,oe_,dv_iqr,dv_median,DEVICE)
        gyro_seg=np.linalg.norm(xrs[os_:oe_,WIN_LEN//2,3:6],axis=-1)
        pred_y=r['dv_pred_ms'][os_-os_:oe_-os_,1]
        idx=min(i,len(norm_vprevs)-1)
        vprev_y=norm_vprevs[idx][:,1]
        err_y=np.abs(pred_y-vprev_y[:len(pred_y)])
        grp=res.get('group','?')
        if 'straight' in grp or grp=='FALSE-ALARM': straight_err.extend(err_y.tolist())
        elif grp=='TURN': turn_err.extend(err_y.tolist())
se=np.mean(straight_err) if straight_err else float('nan')
te=np.mean(turn_err) if turn_err else float('nan')
print(f"  |v_pred_y−v_prev_y| straight:{se:.4f} m/s  (target <0.05)")
print(f"  |v_pred_y−v_prev_y| turns:   {te:.4f} m/s")
if not math.isnan(se):
    if se<0.05: print(f"  ✓ v_prev tracking confirmed — S47-S52 should be fixed")
    elif se<0.15: print(f"  ~ Partial tracking. Better than R19's 0.276 m/s but not at target.")
    else: print(f"  ✗ v_prev tracking poor. L_cvprior not dominating on straight windows.")

print("\n"+"="*65)
print("  SUMMARY vs PREVIOUS RUNS")
print(f"  {'Metric':<28} {'R16':>7} {'R17':>7} {'R19':>7} {'R20':>7}")
print(f"  {'-'*57}")
print(f"  {'v_prev ablation':<28} {'1.00x':>7} {'1.00x':>7} {'1.00x':>7} {f'{ratio1:.2f}x':>7}")
print(f"  {'IMU shuffle':<28} {'0.87x':>7} {'1.02x':>7} {'1.04x':>7} {f'{ratio2:.2f}x':>7}")
print(f"  {'TCN contribution':<28} {'1.00x':>7} {'N/A':>7} {'14.53x':>7} {f'{ratio5:.2f}x':>7}")
print(f"  {'vprev tracking err':<28} {'N/A':>7} {'N/A':>7} {'0.276':>7} {f'{se:.4f}':>7}")
print(f"  {'Mean drift':<28} {'6.80m':>7} {'13.98m':>7} {'8.79m':>7} {f'{np.mean(mdrifts):.2f}m':>7}")
print(f"  {'% under 5m':<28} {'69.5%':>7} {'69.5%':>7} {'59.3%':>7} {f'{100*np.mean(np.array(mdrifts)<5):.1f}%':>7}")
print("="*65)

In [ ]:
import pandas as pd

# CSV for figures notebook
rows = [{'seq_idx': r['seq_idx'], 'drift_m': r['drift_m'],
         'naive_drift_m': r['naive_drift_m'], 'group': r['group'],
         'outage_start': r['outage_start']} for r in all_res]
pd.DataFrame(rows).to_csv(os.path.join(PLOT_DIR, 'val_seq_results.csv'), index=False)
print(f'Saved val_seq_results.csv ({len(rows)} rows)')

# Path arrays for S41, S47, S53
for r in all_res:
    if r['seq_idx'] in [41, 47, 53]:
        np.save(os.path.join(PLOT_DIR, f"path_seq{r['seq_idx']}.npy"), {
            'pos_pred':      r['pos_pred'][:, :2],
            'pos_true':      r['pos_gt'][:, :2],   # R20 uses pos_gt, same as LSTM
            'outage_start':  r['outage_start'],
            'drift_m':       r['drift_m'],
            'naive_drift_m': r['naive_drift_m'],
        })
        print(f"  Saved path_seq{r['seq_idx']}.npy  (drift={r['drift_m']:.2f}m)")